In [1]:
!pip install -q timm safetensors torch torchvision numpy requests matplotlib tqdm pandas psutil

# Federated Learning Under Data Poisoning

Notebook 4 established the clean federated-learning baseline. This notebook keeps that training pipeline fixed and introduces malicious clients.

We load the exact 10 client datasets produced by Notebook 4, use the same MobileNetV4 Conv Small 0.5 initialization, and keep the same federated hyperparameters. The only intended difference between the clean control and the poisoning runs is the malicious modification of selected clients' local training data.

For this reason, the run with malicious fraction $f=0$ acts as an internal control. Its clean-test convergence should (approximately) reproduce the plain FL trajectory from Notebook 4. The notebook compares those trajectories directly when the previous `fl_history.csv` is available.


## What we test

We study two datapoisoning mechanisms.

**Label flipping.** On each malicious client, examples from the source class `circle` are relabeled as the target class `donut`. With the default flip rate of 1.0, every local `circle` training example on a malicious client is changed to `donut`. The image itself is unchanged.

**Backdoor poisoning.** A fixed pixel trigger is added to 20% of each malicious client's local training images, excluding images that already belong to the target class. Those triggered samples are relabeled as `donut`. At evaluation time, the same trigger is applied to clean test images to measure whether it activates the learned backdoor.

For both mechanisms we sweep the malicious-client fraction:

$$f \in \{0.0,\ 0.1,\ 0.3,\ 0.5,\ 1.0\}.$$

With 10 clients, these correspond to 0, 1, 3, 5, and 10 malicious clients.


## Controlled comparison with the clean FL baseline

The clean and poisoned experiments use the same:

- 10 persisted client datasets from Notebook 4,
- Stage 1 MobileNetV4 Conv Small 0.5 checkpoint,
- 50 communication rounds,
- full client participation,
- one local epoch,
- Adam optimizer,
- batch size 128,
- learning rate $10^{-3}$,
- frozen backbone and trainable classifier head,
- sample-weighted FedAvg,
- centralized Stage 1 test split.

The training RNG is reset immediately before the FL rounds in both notebooks. This makes the $f=0$ label-flip control directly comparable to the clean baseline instead of relying only on similar hyperparameters.

The notebook reports the maximum round-by-round difference in clean test accuracy and loss between the two runs. A nonzero difference is therefore visible rather than silently treated as equivalent.


## Imports and experiment configuration


In [2]:
import copy
import csv
import json
import math
import os
import random
import urllib.request
import threading
import time

import psutil
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, TensorDataset
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import timm
import timm.data as timm_data
from safetensors.torch import load_file as load_safetensors
from safetensors.torch import save_file as save_safetensors

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {DEVICE}")

# Stage 1 artifacts produced by the centralized fine-tuning notebook.
STAGE1_OUTPUT_DIR = Path("./outputs")
STAGE1_CONFIG_PATH = STAGE1_OUTPUT_DIR / "config.json"

# Raw QuickDraw data location.
DATA_DIR = Path("./data")

# The saved client datasets from the clean FL pipeline.
CLIENT_DATASET_DIR = Path("./client_datasets")
PLAIN_BASELINE_HISTORY_PATH = Path("./fl_history.csv")

# Output directory for Attack 3.
OUTPUT_DIR = Path("./outputs_attack3_poisoning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "final_models").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "embedding_cache").mkdir(parents=True, exist_ok=True)

# Human-readable label -> QuickDraw dataset class name, as used in .npy filenames.
SELECTED_CLASSES = {
    "line": "line",
    "circle": "circle",
    "square": "square",
    "triangle": "triangle",
    "star": "star",
    "tshirt": "t-shirt",
    "smiley_face": "smiley_face",
    "sun": "sun",
    "cloud": "cloud",
    "donut": "donut",
}
CLASS_NAMES = list(SELECTED_CLASSES.keys())
QUICKDRAW_FILE_NAMES = list(SELECTED_CLASSES.values())
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

# Original Stage 1 split configuration. The clean/triggered test sets are derived from this.
MAX_ITEMS_PER_CLASS = 5_000
TRAIN_PCT = 0.8
VAL_PCT = 0.1
TEST_PCT = 0.1
assert abs(TRAIN_PCT + VAL_PCT + TEST_PCT - 1.0) < 1e-9

# Federated learning configuration.
NUM_CLIENTS = 10
NUM_ROUNDS = 50
CLIENT_FRACTION = 1.0              # all clients participate by default
LOCAL_EPOCHS = 1
LOCAL_BATCH_SIZE = 128
LOCAL_LR = 1e-3
LOCAL_WEIGHT_DECAY = 0.0

# DataLoader configuration. Keep workers disabled for Colab/Jupyter robustness.
FORWARD_BATCH_SIZE = 256
NUM_WORKERS = 0
PIN_MEMORY = DEVICE.type == "cuda"
print(f"DataLoader config: num_workers={NUM_WORKERS}, pin_memory={PIN_MEMORY}")

# Poisoning configuration.
MALICIOUS_FRACTIONS = [0.0, 0.1, 0.3, 0.5, 1.0]
POISON_TYPES = ["label_flip", "backdoor"]

SOURCE_CLASS_NAME = "circle"
TARGET_CLASS_NAME = "donut"
SOURCE_CLASS_IDX = CLASS_TO_IDX[SOURCE_CLASS_NAME]
TARGET_CLASS_IDX = CLASS_TO_IDX[TARGET_CLASS_NAME]

# For label flipping, this fraction of source-class examples on malicious clients is relabeled.
# 1.0 means all source-class examples are changed source -> target.
LABEL_FLIP_RATE = 1.0

# For backdoor poisoning, this fraction of each malicious client's local training data is
# triggered and relabeled as the target class. Target-class images are excluded from the
# poisoned subset by default.
BACKDOOR_POISON_FRACTION = 0.20
BACKDOOR_EXCLUDE_TARGET_FROM_POISONING = True

# Trigger pattern on the 28x28 QuickDraw canvas.
TRIGGER_VALUE = 255
TRIGGER_SIZE = 4
TRIGGER_MARGIN = 1

# ASR threshold used only to summarize "rounds to threshold" in the generated LaTeX table.
ASR_THRESHOLD = 80.0

# Embedding cache.
USE_EMBEDDING_CACHE = True


def start_resource_monitor(device: torch.device, sample_interval_s: float = 0.05) -> Dict:
    """Start scoped wall-time, process-RAM, and CUDA-VRAM monitoring."""
    process = psutil.Process(os.getpid())
    baseline_rss = process.memory_info().rss
    rss_samples = [baseline_rss]
    stop_event = threading.Event()

    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
        baseline_cuda_allocated = torch.cuda.memory_allocated(device)
        baseline_cuda_reserved = torch.cuda.memory_reserved(device)
    else:
        baseline_cuda_allocated = None
        baseline_cuda_reserved = None

    def sample_ram():
        while not stop_event.is_set():
            rss_samples.append(process.memory_info().rss)
            stop_event.wait(sample_interval_s)

    thread = threading.Thread(target=sample_ram, daemon=True)
    thread.start()

    return {
        "process": process,
        "baseline_rss": baseline_rss,
        "rss_samples": rss_samples,
        "stop_event": stop_event,
        "thread": thread,
        "start_time": time.perf_counter(),
        "baseline_cuda_allocated": baseline_cuda_allocated,
        "baseline_cuda_reserved": baseline_cuda_reserved,
    }


def stop_resource_monitor(monitor: Dict, device: torch.device) -> Dict:
    """Stop monitoring and return JSON-serializable runtime and memory statistics."""
    if device.type == "cuda":
        torch.cuda.synchronize(device)

    elapsed_s = time.perf_counter() - monitor["start_time"]
    monitor["stop_event"].set()
    monitor["thread"].join(timeout=1.0)
    monitor["rss_samples"].append(monitor["process"].memory_info().rss)

    peak_rss = max(monitor["rss_samples"])
    baseline_rss = monitor["baseline_rss"]

    stats = {
        "fl_loop_elapsed_s": float(elapsed_s),
        "baseline_process_ram_mib": float(baseline_rss / (1024 ** 2)),
        "peak_process_ram_mib": float(peak_rss / (1024 ** 2)),
        "peak_process_ram_increase_mib": float((peak_rss - baseline_rss) / (1024 ** 2)),
        "ram_sampling_interval_s": 0.05,
        "peak_cuda_vram_allocated_mib": None,
        "peak_cuda_vram_reserved_mib": None,
        "peak_cuda_vram_allocated_increase_mib": None,
        "peak_cuda_vram_reserved_increase_mib": None,
    }

    if device.type == "cuda":
        peak_allocated = torch.cuda.max_memory_allocated(device)
        peak_reserved = torch.cuda.max_memory_reserved(device)
        stats.update({
            "peak_cuda_vram_allocated_mib": float(peak_allocated / (1024 ** 2)),
            "peak_cuda_vram_reserved_mib": float(peak_reserved / (1024 ** 2)),
            "peak_cuda_vram_allocated_increase_mib": float(
                (peak_allocated - monitor["baseline_cuda_allocated"]) / (1024 ** 2)
            ),
            "peak_cuda_vram_reserved_increase_mib": float(
                (peak_reserved - monitor["baseline_cuda_reserved"]) / (1024 ** 2)
            ),
        })

    return stats


Using device: cuda
DataLoader config: num_workers=0, pin_memory=True


## QuickDraw and frozen-backbone utilities

The global clean test set is reconstructed from the same Stage 1 split used in the earlier notebooks.

As in the clean FL baseline, the MobileNetV4 backbone remains frozen. Clean client images and test images are therefore converted to fixed backbone embeddings once. Local federated training operates only on the classifier head.

For the backdoor experiment, triggered images must pass through the backbone because the trigger changes the image itself. Their embeddings are computed separately and may be cached.


In [3]:
def download_quickdraw_dataset(root: Path, class_names: Optional[List[str]] = None):
    """Download the numpy bitmap (.npy) files for the requested QuickDraw class names."""
    root.mkdir(parents=True, exist_ok=True)
    base_url = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/"

    print("Downloading QuickDraw dataset files used by the Stage 1 label space...")
    for class_name in tqdm(class_names):
        fpath = root / f"{class_name}.npy"
        if fpath.exists():
            continue
        url = f"{base_url}{class_name.replace('_', '%20').replace('-', '%2D')}.npy"
        try:
            urllib.request.urlretrieve(url, fpath)
        except Exception as e:
            raise RuntimeError(f"Failed to download class '{class_name}' from {url}: {e}")
    print("Download complete.")



def load_quickdraw_arrays(root: Path, file_names: List[str], max_items_per_class: int):
    """Load raw (N, 784) uint8 bitmaps and integer labels for the selected classes."""
    X_list, y_list = [], []
    print(f"Loading up to {max_items_per_class} examples per class for the Stage 1 split...")
    for label_idx, fname in enumerate(tqdm(file_names)):
        fpath = root / f"{fname}.npy"
        data = np.load(fpath, mmap_mode="r")
        n = min(max_items_per_class, data.shape[0])
        print(f"Label id: {label_idx} - Label name: {CLASS_NAMES[label_idx]} ({fname}).")
        print(f"\tNumber of samples in class: {data.shape[0]} - Loaded {n} samples.")
        X_list.append(np.array(data[:n]))
        y_list.append(np.full(n, label_idx, dtype=np.int64))

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    return X, y



def stratified_split_indices(labels: np.ndarray, train_pct: float, val_pct: float, test_pct: float, seed: int = SEED):
    """Per-class stratified split. Returns index arrays into `labels`."""
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []

    for cls in np.unique(labels):
        cls_idx = np.where(labels == cls)[0]
        rng.shuffle(cls_idx)
        n = len(cls_idx)
        n_train = int(round(n * train_pct))
        n_val = int(round(n * val_pct))
        train_idx.append(cls_idx[:n_train])
        val_idx.append(cls_idx[n_train:n_train + n_val])
        test_idx.append(cls_idx[n_train + n_val:])

    train_idx = np.concatenate(train_idx)
    val_idx = np.concatenate(val_idx)
    test_idx = np.concatenate(test_idx)

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)
    return train_idx, val_idx, test_idx


class QuickDrawRawDataset(Dataset):
    """Raw QuickDraw flattened bitmap dataset with timm preprocessing."""

    def __init__(self, X_flat: np.ndarray, y: np.ndarray, transform):
        self.X_flat = X_flat.astype(np.uint8, copy=False)
        self.y = y.astype(np.int64, copy=False)
        self.transform = transform

    def __len__(self):
        return int(self.X_flat.shape[0])

    def __getitem__(self, idx):
        img_28 = self.X_flat[idx].reshape(28, 28).astype(np.uint8)
        img_rgb = np.stack([img_28, img_28, img_28], axis=-1)
        pil_img = Image.fromarray(img_rgb)
        x = self.transform(pil_img)
        y = int(self.y[idx])
        return x, y


@torch.no_grad()
def embedding_of(model: nn.Module, images_3ch: torch.Tensor) -> torch.Tensor:
    """Return pooled frozen-backbone embeddings for normalized model inputs."""
    feats = model.forward_features(images_3ch)
    pooled = model.forward_head(feats, pre_logits=True)
    return pooled


@torch.no_grad()
def compute_embeddings(
    model: nn.Module,
    X_flat: np.ndarray,
    y: np.ndarray,
    transform,
    batch_size: int = FORWARD_BATCH_SIZE,
    desc: str = "Computing embeddings",
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Precompute frozen-backbone embeddings and labels as CPU tensors."""
    ds = QuickDrawRawDataset(X_flat, y, transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model.eval()
    all_emb, all_y = [], []
    for xb, yb in tqdm(loader, desc=desc):
        xb = xb.to(DEVICE, non_blocking=True)
        emb = embedding_of(model, xb).detach().cpu()
        all_emb.append(emb)
        all_y.append(yb.cpu())

    return torch.cat(all_emb, dim=0), torch.cat(all_y, dim=0)


## Load the exact client population from Notebook 4


This notebook does not construct new clients. It requires client IDs 0 through 9 to be present and stops with an error if the set is incomplete. You can download the files [here](https://drive.google.com/drive/folders/1J1fCvwHqQy22TGQ4WFQs6ODiRj6uJCvy?usp=sharing)

This check is important for attribution. If the client population changed between the clean and poisoning runs, a convergence difference could come from the data distribution rather than from poisoning.


In [4]:
def add_trigger_to_flat_images(
    X_flat: np.ndarray,
    trigger_size: int = TRIGGER_SIZE,
    margin: int = TRIGGER_MARGIN,
    value: int = TRIGGER_VALUE,
) -> np.ndarray:
    """Add a fixed bottom-right square trigger to raw flattened 28x28 uint8 images."""
    X_mod = X_flat.copy().astype(np.uint8)
    imgs = X_mod.reshape(-1, 28, 28)

    r1 = 28 - margin
    r0 = r1 - trigger_size
    c1 = 28 - margin
    c0 = c1 - trigger_size
    imgs[:, r0:r1, c0:c1] = np.uint8(value)

    # Add a tiny L-shape extension to make the trigger less likely to match natural strokes.
    if r0 - 2 >= 0:
        imgs[:, r0 - 2:r0, c0:c1] = np.uint8(value)
    if c0 - 2 >= 0:
        imgs[:, r0:r1, c0 - 2:c0] = np.uint8(value)

    return imgs.reshape(-1, 784)


@dataclass
class LoadedClient:
    client_id: int
    train_X: np.ndarray
    train_y: np.ndarray
    val_X: np.ndarray
    val_y: np.ndarray
    class_counts: np.ndarray
    train_embeddings: Optional[torch.Tensor] = None
    train_labels: Optional[torch.Tensor] = None
    val_embeddings: Optional[torch.Tensor] = None
    val_labels: Optional[torch.Tensor] = None

    @property
    def n_train(self) -> int:
        return int(self.train_y.shape[0])

    @property
    def n_val(self) -> int:
        return int(self.val_y.shape[0])

    @property
    def n_total(self) -> int:
        return self.n_train + self.n_val


@dataclass
class ScenarioTrainData:
    embeddings: torch.Tensor
    labels: torch.Tensor
    n_train: int
    is_malicious: bool
    n_poisoned: int



def load_saved_clients(client_dataset_dir: Path) -> List[LoadedClient]:
    """Load reusable client datasets saved by the clean FL pipeline."""
    if not client_dataset_dir.exists():
        raise FileNotFoundError(
            f"Client dataset directory not found: {client_dataset_dir}\n"
            "Run the clean FL pipeline and save client_datasets first."
        )

    client_files = sorted(client_dataset_dir.glob("client_*.npz"))
    if len(client_files) == 0:
        raise FileNotFoundError(f"No client_*.npz files found in {client_dataset_dir}")

    clients = []
    for path in client_files:
        data = np.load(path)
        stem = path.stem  # client_00
        client_id = int(stem.split("_")[1])
        clients.append(
            LoadedClient(
                client_id=client_id,
                train_X=data["train_X"].astype(np.uint8),
                train_y=data["train_y"].astype(np.int64),
                val_X=data["val_X"].astype(np.uint8),
                val_y=data["val_y"].astype(np.int64),
                class_counts=data["class_counts"].astype(np.int64),
            )
        )

    clients = sorted(clients, key=lambda c: c.client_id)
    expected_ids = list(range(NUM_CLIENTS))
    actual_ids = [c.client_id for c in clients]
    if actual_ids != expected_ids:
        raise RuntimeError(
            f"Expected the exact {NUM_CLIENTS} clients produced by Notebook 4 with ids "
            f"{expected_ids}, but found {actual_ids}. Re-run the revised clean FL notebook."
        )

    print(f"Loaded {len(clients)} clients from {client_dataset_dir}")
    for c in clients:
        readable = ", ".join(
            f"{CLASS_NAMES[i]}:{int(c.class_counts[i])}"
            for i in range(NUM_CLASSES)
            if int(c.class_counts[i]) > 0
        )
        print(f"  Client {c.client_id:02d}: train={c.n_train}, val={c.n_val}, total={c.n_total} | {readable}")
    return clients



def cache_path_for_client_embeddings(client_id: int, split: str) -> Path:
    return OUTPUT_DIR / "embedding_cache" / f"client_{client_id:02d}_{split}_clean.pt"



def cache_path_for_backdoor_client(client_id: int, poison_fraction: float, target_idx: int) -> Path:
    return OUTPUT_DIR / "embedding_cache" / (
        f"client_{client_id:02d}_backdoor_train_"
        f"rate{poison_fraction:.3f}_target{target_idx}.pt"
    )



def precompute_clean_client_embeddings(clients: List[LoadedClient], model: nn.Module, transform):
    """Compute or load clean train/validation embeddings for every client."""
    for client in clients:
        train_cache = cache_path_for_client_embeddings(client.client_id, "train")
        val_cache = cache_path_for_client_embeddings(client.client_id, "val")

        if USE_EMBEDDING_CACHE and train_cache.exists():
            obj = torch.load(train_cache, map_location="cpu")
            client.train_embeddings = obj["embeddings"]
            client.train_labels = obj["labels"]
        else:
            emb, labels = compute_embeddings(
                model,
                client.train_X,
                client.train_y,
                transform,
                desc=f"Client {client.client_id:02d} clean train embeddings",
            )
            client.train_embeddings = emb
            client.train_labels = labels
            if USE_EMBEDDING_CACHE:
                torch.save({"embeddings": emb, "labels": labels}, train_cache)

        if USE_EMBEDDING_CACHE and val_cache.exists():
            obj = torch.load(val_cache, map_location="cpu")
            client.val_embeddings = obj["embeddings"]
            client.val_labels = obj["labels"]
        else:
            emb, labels = compute_embeddings(
                model,
                client.val_X,
                client.val_y,
                transform,
                desc=f"Client {client.client_id:02d} clean val embeddings",
            )
            client.val_embeddings = emb
            client.val_labels = labels
            if USE_EMBEDDING_CACHE:
                torch.save({"embeddings": emb, "labels": labels}, val_cache)


## The unchanged FL training rule

Every selected client receives the current global classifier head, performs one local epoch with Adam, and returns the resulting head parameters.

The server uses sample-weighted FedAvg:

$$\theta_{t+1}
=
\sum_{k \in S_t}
\frac{n_k}{\sum_{j \in S_t}n_j}
\theta_{t+1}^{(k)}.$$

There is no robust aggregation or attack detection in this notebook. Malicious and honest updates are treated identically by the server.


In [5]:
class LinearHead(nn.Module):
    """Standalone classifier head trained on frozen backbone embeddings."""

    def __init__(self, in_features: int, num_classes: int):
        super().__init__()
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        return self.classifier(embeddings)



def make_head_from_model(model: nn.Module) -> LinearHead:
    classifier = model.get_classifier()
    if not isinstance(classifier, nn.Linear):
        raise TypeError(f"Expected a linear classifier head, got: {type(classifier)}")
    head = LinearHead(classifier.in_features, classifier.out_features)
    head.classifier.load_state_dict(copy.deepcopy(classifier.state_dict()))
    return head



def train_local_head(global_head: LinearHead, embeddings: torch.Tensor, labels: torch.Tensor) -> LinearHead:
    """Train a local head on one client's embeddings using Adam."""
    local_head = copy.deepcopy(global_head).to(DEVICE)
    local_head.train()

    ds = TensorDataset(embeddings, labels)
    loader = DataLoader(ds, batch_size=LOCAL_BATCH_SIZE, shuffle=True)

    optimizer = torch.optim.Adam(local_head.parameters(), lr=LOCAL_LR, weight_decay=LOCAL_WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    for _epoch in range(LOCAL_EPOCHS):
        for emb, yb in loader:
            emb = emb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = local_head(emb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    return local_head.cpu()



def fedavg_heads(local_heads: List[LinearHead], weights: List[int]) -> LinearHead:
    """Weighted FedAvg over classifier heads."""
    assert len(local_heads) == len(weights) and len(local_heads) > 0
    total_weight = float(sum(weights))

    avg_head = copy.deepcopy(local_heads[0])
    avg_state = avg_head.state_dict()
    for key in avg_state:
        avg_state[key] = torch.zeros_like(avg_state[key])

    for head, weight in zip(local_heads, weights):
        state = head.state_dict()
        coeff = float(weight) / total_weight
        for key in avg_state:
            avg_state[key] += coeff * state[key]

    avg_head.load_state_dict(avg_state)
    return avg_head


@torch.no_grad()
def evaluate_head_on_embeddings(
    head: LinearHead,
    embeddings: torch.Tensor,
    labels: torch.Tensor,
    batch_size: int = 2048,
) -> Tuple[float, float]:
    """Evaluate head on embeddings. Returns average loss and accuracy percentage."""
    head_eval = copy.deepcopy(head).to(DEVICE)
    head_eval.eval()
    ds = TensorDataset(embeddings, labels)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    criterion = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    total_correct = 0
    total = 0
    for emb, yb in loader:
        emb = emb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        logits = head_eval(emb)
        total_loss += float(criterion(logits, yb).item())
        total_correct += int((logits.argmax(dim=1) == yb).sum().item())
        total += int(yb.numel())

    return total_loss / max(total, 1), 100.0 * total_correct / max(total, 1)


@torch.no_grad()
def compute_target_prediction_rate(
    head: LinearHead,
    embeddings: torch.Tensor,
    labels: torch.Tensor,
    mask: torch.Tensor,
    target_idx: int,
    batch_size: int = 2048,
) -> float:
    """Fraction of masked examples predicted as target_idx."""
    idx = torch.where(mask.cpu())[0]
    if idx.numel() == 0:
        return float("nan")

    head_eval = copy.deepcopy(head).to(DEVICE)
    head_eval.eval()

    emb_sel = embeddings[idx]
    y_dummy = labels[idx]
    ds = TensorDataset(emb_sel, y_dummy)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    target_preds = 0
    total = 0
    for emb, _yb in loader:
        emb = emb.to(DEVICE, non_blocking=True)
        logits = head_eval(emb)
        preds = logits.argmax(dim=1).detach().cpu()
        target_preds += int((preds == int(target_idx)).sum().item())
        total += int(preds.numel())

    return 100.0 * target_preds / max(total, 1)


## Construct the poisoned local training data

Malicious clients are selected deterministically from the actual persisted client IDs.

For label flipping, only labels are modified, so the clean frozen-backbone embeddings can be reused.

For the backdoor experiment, selected raw client images are modified with the trigger and relabeled to the target class. Because the pixels change, the triggered images are forwarded through MobileNetV4 to compute new embeddings. Honest clients continue to use their original clean embeddings.

Only the local **training** split is poisoned. Client validation data and the centralized clean test set remain unchanged.


In [6]:
def choose_malicious_clients(fraction: float, client_ids: List[int], seed: int = SEED) -> List[int]:
    """Choose a deterministic subset of the actual persisted client IDs."""
    if fraction <= 0:
        return []
    n_clients = len(client_ids)
    n_malicious = int(round(fraction * n_clients))
    n_malicious = max(1, min(n_clients, n_malicious))
    rng = np.random.default_rng(seed + int(round(fraction * 1000)) + 777)
    ids = sorted(
        int(x)
        for x in rng.choice(np.asarray(client_ids, dtype=np.int64), size=n_malicious, replace=False)
    )
    return ids



def make_label_flip_training_data(
    client: LoadedClient,
    is_malicious: bool,
    source_idx: int,
    target_idx: int,
    flip_rate: float,
    seed: int,
) -> ScenarioTrainData:
    """Use clean embeddings but relabel source-class examples for malicious clients."""
    labels = client.train_labels.clone()
    n_poisoned = 0
    if is_malicious:
        source_positions = torch.where(labels == int(source_idx))[0].cpu().numpy()
        if len(source_positions) > 0:
            rng = np.random.default_rng(seed + 9000 + client.client_id)
            n_to_flip = int(round(float(flip_rate) * len(source_positions)))
            n_to_flip = max(1, min(len(source_positions), n_to_flip)) if flip_rate > 0 else 0
            chosen = rng.choice(source_positions, size=n_to_flip, replace=False) if n_to_flip > 0 else []
            labels[chosen] = int(target_idx)
            n_poisoned = int(n_to_flip)

    return ScenarioTrainData(
        embeddings=client.train_embeddings,
        labels=labels,
        n_train=client.n_train,
        is_malicious=is_malicious,
        n_poisoned=n_poisoned,
    )



def make_backdoor_training_data(
    client: LoadedClient,
    is_malicious: bool,
    model: nn.Module,
    transform,
    poison_fraction: float,
    target_idx: int,
    seed: int,
) -> ScenarioTrainData:
    """Create triggered/relabelled local data for malicious clients, clean data otherwise."""
    if not is_malicious or poison_fraction <= 0:
        return ScenarioTrainData(
            embeddings=client.train_embeddings,
            labels=client.train_labels.clone(),
            n_train=client.n_train,
            is_malicious=is_malicious,
            n_poisoned=0,
        )

    cache = cache_path_for_backdoor_client(client.client_id, poison_fraction, target_idx)
    if USE_EMBEDDING_CACHE and cache.exists():
        obj = torch.load(cache, map_location="cpu")
        return ScenarioTrainData(
            embeddings=obj["embeddings"],
            labels=obj["labels"],
            n_train=client.n_train,
            is_malicious=True,
            n_poisoned=int(obj["n_poisoned"]),
        )

    X_poisoned = client.train_X.copy().astype(np.uint8)
    y_poisoned = client.train_y.copy().astype(np.int64)

    if BACKDOOR_EXCLUDE_TARGET_FROM_POISONING:
        candidates = np.where(y_poisoned != int(target_idx))[0]
    else:
        candidates = np.arange(len(y_poisoned))

    rng = np.random.default_rng(seed + 10000 + client.client_id)
    n_to_poison = int(round(poison_fraction * len(y_poisoned)))
    n_to_poison = min(n_to_poison, len(candidates))
    n_to_poison = max(1, n_to_poison) if len(candidates) > 0 else 0

    if n_to_poison > 0:
        poison_idx = rng.choice(candidates, size=n_to_poison, replace=False)
        X_poisoned[poison_idx] = add_trigger_to_flat_images(X_poisoned[poison_idx])
        y_poisoned[poison_idx] = int(target_idx)

    emb, labels = compute_embeddings(
        model,
        X_poisoned,
        y_poisoned,
        transform,
        desc=f"Client {client.client_id:02d} backdoor-poisoned train embeddings",
    )

    if USE_EMBEDDING_CACHE:
        torch.save(
            {"embeddings": emb, "labels": labels, "n_poisoned": int(n_to_poison)},
            cache,
        )

    return ScenarioTrainData(
        embeddings=emb,
        labels=labels,
        n_train=client.n_train,
        is_malicious=True,
        n_poisoned=int(n_to_poison),
    )



def prepare_scenario_training_data(
    poison_type: str,
    clients: List[LoadedClient],
    malicious_ids: List[int],
    model: nn.Module,
    transform,
    seed: int,
) -> Dict[int, ScenarioTrainData]:
    """Prepare per-client training tensors for a poisoning scenario."""
    malicious_set = set(malicious_ids)
    scenario = {}

    for client in clients:
        is_malicious = client.client_id in malicious_set
        if poison_type == "label_flip":
            scenario[client.client_id] = make_label_flip_training_data(
                client,
                is_malicious=is_malicious,
                source_idx=SOURCE_CLASS_IDX,
                target_idx=TARGET_CLASS_IDX,
                flip_rate=LABEL_FLIP_RATE,
                seed=seed,
            )
        elif poison_type == "backdoor":
            scenario[client.client_id] = make_backdoor_training_data(
                client,
                is_malicious=is_malicious,
                model=model,
                transform=transform,
                poison_fraction=BACKDOOR_POISON_FRACTION,
                target_idx=TARGET_CLASS_IDX,
                seed=seed,
            )
        else:
            raise ValueError(f"Unsupported poison_type: {poison_type}")

    return scenario


## Metrics

Two outcomes are tracked after every communication round.

**Clean test accuracy** measures whether poisoning damages ordinary model performance on the unchanged Stage 1 test set.

**Attack success rate (ASR)** depends on the poisoning mechanism:

- For label flipping, ASR is the percentage of clean source-class (`circle`) test images predicted as the target class (`donut`).
- For the backdoor, ASR is the percentage of triggered non-target test images predicted as `donut`.

The $f=0$ ASR is a background prediction rate, not an attack result. It shows how often the clean model already produces the target prediction under the relevant evaluation condition.

We also record the number of poisoned local samples, the malicious client IDs, rounds required to reach the configured ASR threshold, scenario runtime, peak process RAM, and CUDA VRAM when available.


In [7]:
def save_full_model_with_head(
    stage1_state_dict: Dict[str, torch.Tensor],
    head: LinearHead,
    output_path: Path,
):
    """Save a full model state dict by replacing only the classifier head in Stage 1."""
    state = {k: v.detach().cpu().contiguous().clone() for k, v in stage1_state_dict.items()}
    head_state = head.state_dict()

    # LinearHead has keys classifier.weight and classifier.bias, same as timm model here.
    for key, value in head_state.items():
        if key not in state:
            raise KeyError(f"Head key {key} not present in Stage 1 model state dict.")
        state[key] = value.detach().cpu().contiguous().clone()

    save_safetensors(state, str(output_path))



def rounds_to_asr_threshold(history: List[Dict], threshold: float = ASR_THRESHOLD) -> Optional[int]:
    """First round where ASR reaches threshold. Returns None if never reached."""
    for row in history:
        if row["round"] == 0:
            continue
        asr = row.get("asr", float("nan"))
        if not math.isnan(asr) and asr >= threshold:
            return int(row["round"])
    return None



def run_poisoning_experiment(
    poison_type: str,
    malicious_fraction: float,
    clients: List[LoadedClient],
    base_head: LinearHead,
    model: nn.Module,
    transform,
    test_embeddings: torch.Tensor,
    test_labels: torch.Tensor,
    triggered_test_embeddings: Optional[torch.Tensor],
    stage1_state_dict: Dict[str, torch.Tensor],
) -> Tuple[List[Dict], Dict]:
    """Run one poisoning scenario for a fixed poison type and malicious-client fraction."""
    malicious_ids = choose_malicious_clients(
        malicious_fraction, [c.client_id for c in clients], seed=SEED
    )
    print("\n" + "=" * 88)
    print(f"Attack 3 | poison_type={poison_type} | f={malicious_fraction:.2f} | malicious_ids={malicious_ids}")
    print("=" * 88)

    scenario_train = prepare_scenario_training_data(
        poison_type,
        clients,
        malicious_ids,
        model,
        transform,
        seed=SEED + int(round(malicious_fraction * 1000)),
    )

    n_poisoned_total = sum(scenario_train[c.client_id].n_poisoned for c in clients)
    n_malicious_train_total = sum(c.n_train for c in clients if c.client_id in set(malicious_ids))
    print(f"Malicious clients: {malicious_ids}")
    print(f"Total poisoned local train samples: {n_poisoned_total}")
    print(f"Total malicious-client train samples: {n_malicious_train_total}")

    global_head = copy.deepcopy(base_head)
    history: List[Dict] = []

    source_mask = test_labels == int(SOURCE_CLASS_IDX)
    non_target_mask = test_labels != int(TARGET_CLASS_IDX)

    def record(round_idx: int):
        clean_loss, clean_acc = evaluate_head_on_embeddings(global_head, test_embeddings, test_labels)

        if poison_type == "label_flip":
            asr = compute_target_prediction_rate(
                global_head,
                test_embeddings,
                test_labels,
                source_mask,
                TARGET_CLASS_IDX,
            )
            source_clean_to_target_asr = asr
            triggered_non_target_asr = float("nan")
        elif poison_type == "backdoor":
            if triggered_test_embeddings is None:
                raise RuntimeError("triggered_test_embeddings is required for backdoor evaluation.")
            asr = compute_target_prediction_rate(
                global_head,
                triggered_test_embeddings,
                test_labels,
                non_target_mask,
                TARGET_CLASS_IDX,
            )
            source_clean_to_target_asr = compute_target_prediction_rate(
                global_head,
                test_embeddings,
                test_labels,
                source_mask,
                TARGET_CLASS_IDX,
            )
            triggered_non_target_asr = asr
        else:
            raise ValueError(poison_type)

        row = {
            "poison_type": poison_type,
            "malicious_fraction": float(malicious_fraction),
            "num_malicious_clients": int(len(malicious_ids)),
            "malicious_clients": ",".join(map(str, malicious_ids)) if malicious_ids else "",
            "round": int(round_idx),
            "clean_test_loss": float(clean_loss),
            "clean_test_acc": float(clean_acc),
            "asr": float(asr),
            "source_clean_to_target_asr": float(source_clean_to_target_asr),
            "triggered_non_target_asr": float(triggered_non_target_asr),
            "n_poisoned_total": int(n_poisoned_total),
            "n_malicious_train_total": int(n_malicious_train_total),
        }
        history.append(row)
        print(
            f"[{poison_type} | f={malicious_fraction:.2f} | round {round_idx:02d}] "
            f"clean test loss={clean_loss:.4f}, acc={clean_acc:.2f}% | "
            f"ASR={asr:.2f}%"
        )

    record(0)

    # Controlled comparison: every scenario starts local optimization from the
    # same training RNG state. Notebook 4 resets the RNG at the same point.
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    rng_round = np.random.default_rng(SEED + 5000)
    num_selected = max(1, int(round(CLIENT_FRACTION * len(clients))))

    fl_monitor = start_resource_monitor(DEVICE)
    round_finetune_times = []

    for round_idx in range(1, NUM_ROUNDS + 1):
        round_finetune_start = time.perf_counter()
        if num_selected == len(clients):
            selected_clients = clients
        else:
            selected_ids = rng_round.choice(len(clients), size=num_selected, replace=False)
            selected_clients = [clients[int(i)] for i in selected_ids]

        local_heads = []
        local_weights = []
        for client in selected_clients:
            train_data = scenario_train[client.client_id]
            local_head = train_local_head(global_head, train_data.embeddings, train_data.labels)
            local_heads.append(local_head)
            local_weights.append(train_data.n_train)

        global_head = fedavg_heads(local_heads, local_weights)
        round_finetune_times.append(time.perf_counter() - round_finetune_start)
        record(round_idx)

    resource_stats = stop_resource_monitor(fl_monitor, DEVICE)
    federated_finetuning_time_s = float(sum(round_finetune_times))

    print(
        f"Federated fine-tuning + aggregation time: {federated_finetuning_time_s:.2f} s | "
        f"full loop including evaluation: {resource_stats['fl_loop_elapsed_s']:.2f} s"
    )
    print(
        f"Peak process RAM: {resource_stats['peak_process_ram_mib']:.1f} MiB "
        f"(+{resource_stats['peak_process_ram_increase_mib']:.1f} MiB)"
    )
    if resource_stats["peak_cuda_vram_allocated_mib"] is not None:
        print(
            f"Peak CUDA VRAM allocated: {resource_stats['peak_cuda_vram_allocated_mib']:.1f} MiB | "
            f"reserved: {resource_stats['peak_cuda_vram_reserved_mib']:.1f} MiB"
        )

    # Save final full model for this scenario.
    frac_tag = f"f{int(round(malicious_fraction * 100)):02d}"
    model_path = OUTPUT_DIR / "final_models" / f"attack3_{poison_type}_{frac_tag}.safetensors"
    save_full_model_with_head(stage1_state_dict, global_head, model_path)

    summary = {
        "poison_type": poison_type,
        "malicious_fraction": float(malicious_fraction),
        "num_malicious_clients": int(len(malicious_ids)),
        "malicious_clients": malicious_ids,
        "n_poisoned_total": int(n_poisoned_total),
        "n_malicious_train_total": int(n_malicious_train_total),
        "initial": history[0],
        "final": history[-1],
        "best_asr": max(row["asr"] for row in history),
        "round_to_asr_threshold": rounds_to_asr_threshold(history, ASR_THRESHOLD),
        "resource_metrics": {
            "federated_finetuning_time_s": federated_finetuning_time_s,
            **resource_stats,
        },
        "saved_model": str(model_path),
    }
    return history, summary



def save_history_csv(history: List[Dict], output_path: Path):
    if len(history) == 0:
        return
    with open(output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)
    print(f"Saved history to: {output_path}")



def save_summary_json(summary: List[Dict], output_path: Path):
    with open(output_path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"Saved summary to: {output_path}")



def save_latex_table(poison_type: str, summaries: List[Dict], output_path: Path):
    """Save a LaTeX table for Attack 3 for one poison type."""
    title = "label-flipping" if poison_type == "label_flip" else "backdoor trigger"

    lines = []
    lines.append(f"% Attack 3 {title} table generated by flta_attack3_poisoning.py")
    lines.append("\\begin{table}[h]")
    lines.append("\\centering")
    lines.append(f"\\caption{{Attack 3: undefended FedAvg under {title} poisoning.}}")
    lines.append(f"\\label{{tab:attack3-{poison_type.replace('_', '-')}}}")
    lines.append("\\begin{tabular}{lccc}")
    lines.append("\\toprule")
    lines.append("Malicious fraction $f$ & Clean test acc. (\\%) & Attack success rate (\\%) & Rounds to ASR $\\geq 80\\%$ \\\\")
    lines.append("\\midrule")

    for s in summaries:
        f = float(s["malicious_fraction"])
        label = "0\\% (no attack)" if f == 0.0 else f"{int(round(f * 100))}\\%"
        final_acc = s["final"]["clean_test_acc"]
        final_asr = s["final"]["asr"]
        r_asr = s["round_to_asr_threshold"]

        if f == 0.0:
            asr_text = "---"
            rounds_text = "---"
        else:
            asr_text = f"{final_asr:.2f}"
            rounds_text = "not reached" if r_asr is None else str(r_asr)

        lines.append(f"{label} & {final_acc:.2f} & {asr_text} & {rounds_text} \\\\")

    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    lines.append("\\end{table}")
    output_path.write_text("\n".join(lines))
    print(f"Saved LaTeX table to: {output_path}")



def save_curves(poison_type: str, history: List[Dict], output_dir: Path):
    """Save clean accuracy and ASR curves for a poison type."""
    df = pd.DataFrame(history)
    fractions = sorted(df["malicious_fraction"].unique())

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    for f in fractions:
        sub = df[df["malicious_fraction"] == f].sort_values("round")
        label = f"f={int(round(float(f) * 100))}%"
        axes[0].plot(sub["round"], sub["clean_test_acc"], marker="o", markersize=3, label=label)
        axes[1].plot(sub["round"], sub["asr"], marker="o", markersize=3, label=label)

    title = "Label-flipping poisoning" if poison_type == "label_flip" else "Backdoor trigger poisoning"
    axes[0].set_title(f"{title}: clean test accuracy")
    axes[0].set_xlabel("Round")
    axes[0].set_ylabel("Clean test accuracy (%)")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].set_title(f"{title}: attack success rate")
    axes[1].set_xlabel("Round")
    axes[1].set_ylabel("ASR (%)")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    fig.tight_layout()
    png_path = output_dir / f"attack3_{poison_type}_curves.png"
    pdf_path = output_dir / f"attack3_{poison_type}_curves.pdf"
    fig.savefig(png_path, dpi=200)
    fig.savefig(pdf_path)
    plt.close(fig)
    print(f"Saved curves to: {png_path}")


def compare_clean_control_to_plain_baseline(
    poisoning_history: List[Dict],
    baseline_history_path: Path,
) -> Optional[Dict]:
    """Compare the f=0 poisoning control against Notebook 4 round by round."""
    if not baseline_history_path.exists():
        print(
            f"Plain baseline history not found at {baseline_history_path}. "
            "Skipping the round-by-round control comparison."
        )
        return None

    baseline = pd.read_csv(baseline_history_path)
    control = pd.DataFrame(poisoning_history)

    required_baseline = {"round", "test_loss", "test_acc"}
    if not required_baseline.issubset(baseline.columns):
        raise RuntimeError(
            f"Baseline history is missing required columns: "
            f"{sorted(required_baseline - set(baseline.columns))}"
        )

    merged = baseline[["round", "test_loss", "test_acc"]].merge(
        control[["round", "clean_test_loss", "clean_test_acc"]],
        on="round",
        how="inner",
        validate="one_to_one",
    )
    if len(merged) != NUM_ROUNDS + 1:
        raise RuntimeError(
            f"Expected {NUM_ROUNDS + 1} aligned rounds, found {len(merged)}."
        )

    acc_diff = (merged["test_acc"] - merged["clean_test_acc"]).abs()
    loss_diff = (merged["test_loss"] - merged["clean_test_loss"]).abs()

    comparison = {
        "n_compared_rounds": int(len(merged)),
        "max_abs_test_accuracy_difference_percentage_points": float(acc_diff.max()),
        "mean_abs_test_accuracy_difference_percentage_points": float(acc_diff.mean()),
        "max_abs_test_loss_difference": float(loss_diff.max()),
        "mean_abs_test_loss_difference": float(loss_diff.mean()),
    }

    print("\\nClean-control comparison with Notebook 4:")
    print(
        "  max |test accuracy difference|: "
        f"{comparison['max_abs_test_accuracy_difference_percentage_points']:.6f} percentage points"
    )
    print(
        "  max |test loss difference|: "
        f"{comparison['max_abs_test_loss_difference']:.8f}"
    )
    return comparison


## Run the poisoning sweep

Each scenario starts from the same Stage 1 classifier head. Poisoning is therefore evaluated independently for every malicious fraction instead of continuing from a previous poisoned run.

For each poisoning type and malicious fraction, the notebook:

1. selects the malicious clients,
2. constructs their poisoned local training tensors,
3. resets the local-training RNG,
4. runs 50 rounds of ordinary FedAvg,
5. evaluates clean accuracy and ASR after every round,
6. saves the final model and complete history.

The $f=0$ label-flip scenario additionally compares its clean-test trajectory with the plain FL history from Notebook 4.


In [8]:
def main():
    print(f"Number of classes: {NUM_CLASSES}")
    print(CLASS_NAMES)
    print(f"Source class: {SOURCE_CLASS_NAME} ({SOURCE_CLASS_IDX})")
    print(f"Target class: {TARGET_CLASS_NAME} ({TARGET_CLASS_IDX})")

    if not STAGE1_CONFIG_PATH.exists():
        raise FileNotFoundError(f"Could not find {STAGE1_CONFIG_PATH}. Run Stage 1 first.")

    with open(STAGE1_CONFIG_PATH, "r") as f:
        stage1_config = json.load(f)

    MODEL_NAME = stage1_config["model_name"]
    EXPECTED_MODEL_NAME = "mobilenetv4_conv_small_050.e3000_r224_in1k"
    assert MODEL_NAME == EXPECTED_MODEL_NAME, (
        f"This notebook expects the FLTA Small MobileNetV4 checkpoint ({EXPECTED_MODEL_NAME}), "
        f"but config.json specifies {MODEL_NAME}."
    )
    STAGE1_CLASS_NAMES = stage1_config["class_names"]
    STAGE1_QUICKDRAW_FILE_NAMES = stage1_config["quickdraw_file_names"]
    IMG_SIZE = stage1_config["input_size"][-1]
    MEAN = stage1_config["mean"]
    STD = stage1_config["std"]
    STAGE1_NUM_CLASSES = stage1_config["num_classes"]

    assert STAGE1_CLASS_NAMES == CLASS_NAMES, "Stage 1 class order mismatch."
    assert STAGE1_QUICKDRAW_FILE_NAMES == QUICKDRAW_FILE_NAMES, "Stage 1 QuickDraw filename order mismatch."
    assert STAGE1_NUM_CLASSES == NUM_CLASSES

    candidate_ckpts = sorted(STAGE1_OUTPUT_DIR.glob("*.safetensors"))
    if len(candidate_ckpts) == 0:
        raise FileNotFoundError(f"No .safetensors checkpoint found in {STAGE1_OUTPUT_DIR}.")
    STAGE1_CKPT_PATH = candidate_ckpts[0]

    print(f"Stage 1 model: {MODEL_NAME}")
    print(f"Stage 1 checkpoint: {STAGE1_CKPT_PATH}")

    global_model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
    num_model_params = sum(p.numel() for p in global_model.parameters())
    print(f"Model parameters: {num_model_params:,}")
    stage1_state_dict = load_safetensors(str(STAGE1_CKPT_PATH))
    missing, unexpected = global_model.load_state_dict(stage1_state_dict, strict=True)
    print(f"Loaded Stage 1 weights. Missing: {missing} | Unexpected: {unexpected}")

    global_model.to(DEVICE)
    global_model.eval()
    for p in global_model.parameters():
        p.requires_grad = False

    classifier = global_model.get_classifier()
    print(f"Classifier head: {classifier}")

    data_config = {
        "input_size": (3, IMG_SIZE, IMG_SIZE),
        "mean": MEAN,
        "std": STD,
    }
    eval_transform = timm_data.create_transform(**data_config, is_training=False)
    print(eval_transform)

    # -------------------------------------------------------------------------
    # Load saved clients and precompute clean client embeddings.
    # -------------------------------------------------------------------------
    clients = load_saved_clients(CLIENT_DATASET_DIR)
    precompute_clean_client_embeddings(clients, global_model, eval_transform)

    # -------------------------------------------------------------------------
    # Build original Stage 1 clean test set and triggered test set.
    # -------------------------------------------------------------------------
    download_quickdraw_dataset(DATA_DIR, QUICKDRAW_FILE_NAMES)
    X_all, y_all = load_quickdraw_arrays(DATA_DIR, QUICKDRAW_FILE_NAMES, MAX_ITEMS_PER_CLASS)
    _train_idx, _val_idx, test_idx = stratified_split_indices(y_all, TRAIN_PCT, VAL_PCT, TEST_PCT, seed=SEED)
    X_test = X_all[test_idx]
    y_test = y_all[test_idx]

    test_cache = OUTPUT_DIR / "embedding_cache" / "global_test_clean.pt"
    triggered_test_cache = OUTPUT_DIR / "embedding_cache" / f"global_test_triggered_target{TARGET_CLASS_IDX}.pt"

    if USE_EMBEDDING_CACHE and test_cache.exists():
        obj = torch.load(test_cache, map_location="cpu")
        test_embeddings = obj["embeddings"]
        test_labels = obj["labels"]
    else:
        test_embeddings, test_labels = compute_embeddings(
            global_model,
            X_test,
            y_test,
            eval_transform,
            desc="Precomputing clean global test embeddings",
        )
        if USE_EMBEDDING_CACHE:
            torch.save({"embeddings": test_embeddings, "labels": test_labels}, test_cache)

    X_test_triggered = add_trigger_to_flat_images(X_test)
    if USE_EMBEDDING_CACHE and triggered_test_cache.exists():
        obj = torch.load(triggered_test_cache, map_location="cpu")
        triggered_test_embeddings = obj["embeddings"]
        triggered_test_labels = obj["labels"]
    else:
        triggered_test_embeddings, triggered_test_labels = compute_embeddings(
            global_model,
            X_test_triggered,
            y_test,
            eval_transform,
            desc="Precomputing triggered global test embeddings",
        )
        if USE_EMBEDDING_CACHE:
            torch.save({"embeddings": triggered_test_embeddings, "labels": triggered_test_labels}, triggered_test_cache)

    assert torch.equal(test_labels, triggered_test_labels)

    # -------------------------------------------------------------------------
    # Base head from Stage 1 checkpoint.
    # -------------------------------------------------------------------------
    base_head = make_head_from_model(global_model)

    # -------------------------------------------------------------------------
    # Run poisoning sweeps.
    # -------------------------------------------------------------------------
    full_config = {
        "model_name": MODEL_NAME,
        "stage1_checkpoint": str(STAGE1_CKPT_PATH),
        "client_dataset_dir": str(CLIENT_DATASET_DIR),
        "plain_baseline_history": str(PLAIN_BASELINE_HISTORY_PATH),
        "class_names": CLASS_NAMES,
        "source_class": SOURCE_CLASS_NAME,
        "target_class": TARGET_CLASS_NAME,
        "num_rounds": NUM_ROUNDS,
        "local_optimizer": "Adam",
        "local_epochs": LOCAL_EPOCHS,
        "local_batch_size": LOCAL_BATCH_SIZE,
        "local_lr": LOCAL_LR,
        "label_flip_rate": LABEL_FLIP_RATE,
        "backdoor_poison_fraction": BACKDOOR_POISON_FRACTION,
        "malicious_fractions": MALICIOUS_FRACTIONS,
        "asr_threshold": ASR_THRESHOLD,
        "trigger": {
            "trigger_size": TRIGGER_SIZE,
            "trigger_margin": TRIGGER_MARGIN,
            "trigger_value": TRIGGER_VALUE,
        },
    }
    with open(OUTPUT_DIR / "attack3_config.json", "w") as f:
        json.dump(full_config, f, indent=2)

    for poison_type in POISON_TYPES:
        all_history: List[Dict] = []
        summaries: List[Dict] = []
        for f_malicious in MALICIOUS_FRACTIONS:
            history, summary = run_poisoning_experiment(
                poison_type=poison_type,
                malicious_fraction=float(f_malicious),
                clients=clients,
                base_head=base_head,
                model=global_model,
                transform=eval_transform,
                test_embeddings=test_embeddings,
                test_labels=test_labels,
                triggered_test_embeddings=triggered_test_embeddings,
                stage1_state_dict=stage1_state_dict,
            )
            if poison_type == "label_flip" and float(f_malicious) == 0.0:
                summary["plain_baseline_comparison"] = compare_clean_control_to_plain_baseline(
                    history, PLAIN_BASELINE_HISTORY_PATH
                )

            all_history.extend(history)
            summaries.append(summary)

        save_history_csv(all_history, OUTPUT_DIR / f"attack3_{poison_type}_history.csv")
        save_summary_json(summaries, OUTPUT_DIR / f"attack3_{poison_type}_summary.json")
        save_latex_table(poison_type, summaries, OUTPUT_DIR / f"attack3_{poison_type}_latex_table.txt")
        save_curves(poison_type, all_history, OUTPUT_DIR)

        print("\n" + "=" * 88)
        print(f"Summary for poison_type={poison_type}")
        print("=" * 88)
        for s in summaries:
            f = s["malicious_fraction"]
            final = s["final"]
            print(
                f"f={f:.2f} | clean_acc={final['clean_test_acc']:.2f}% | "
                f"ASR={final['asr']:.2f}% | "
                f"round_to_ASR{ASR_THRESHOLD:.0f}={s['round_to_asr_threshold']} | "
                f"malicious={s['malicious_clients']}"
            )

    print("\nAttack 3 complete. Outputs saved to:", OUTPUT_DIR)


if __name__ == "__main__":
    main()


Number of classes: 10
['line', 'circle', 'square', 'triangle', 'star', 'tshirt', 'smiley_face', 'sun', 'cloud', 'donut']
Source class: circle (1)
Target class: donut (9)
Stage 1 model: mobilenetv4_conv_small_050.e3000_r224_in1k
Stage 1 checkpoint: outputs/mobilenetv4_small_quickdraw10.safetensors
Model parameters: 970,762
Loaded Stage 1 weights. Missing: [] | Unexpected: []
Classifier head: Linear(in_features=1280, out_features=10, bias=True)
Compose(
    Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.5000, 0.5000, 0.5000]), std=tensor([0.5000, 0.5000, 0.5000]))
)
Loaded 10 clients from client_datasets
  Client 00: train=914, val=101, total=1015 | line:152, circle:61, square:50, triangle:21, star:21, tshirt:151, smiley_face:110, cloud:48, donut:401
  Client 01: train=1701, val=189, total=1890 | line:250, square:182, triangle:466, star:120, smiley_face:74, sun:669, cloud:20, donut:1

Client 00 clean train embeddings:   0%|          | 0/4 [00:00<?, ?it/s]

Client 00 clean val embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

Client 01 clean train embeddings:   0%|          | 0/7 [00:00<?, ?it/s]

Client 01 clean val embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

...

Malicious clients: [0, 1, 2, 8, 9]
Total poisoned local train samples: 1234
Total malicious-client train samples: 6168
[backdoor | f=0.50 | round 00] clean test loss=0.0925, acc=97.66% | ASR=0.33%
[backdoor | f=0.50 | round 01] clean test loss=0.0937, acc=97.66% | ASR=0.47%
[backdoor | f=0.50 | round 02] clean test loss=0.0956, acc=97.64% | ASR=0.76%
[backdoor | f=0.50 | round 03] clean test loss=0.0973, acc=97.62% | ASR=1.24%
[backdoor | f=0.50 | round 04] clean test loss=0.0985, acc=97.62% | ASR=2.62%
[backdoor | f=0.50 | round 05] clean test loss=0.0997, acc=97.60% | ASR=4.31%
[backdoor | f=0.50 | round 06] clean test loss=0.1001, acc=97.56% | ASR=7.51%
[backdoor | f=0.50 | round 07] clean test loss=0.1012, acc=97.54% | ASR=10.53%
[backdoor | f=0.50 | round 08] clean test loss=0.1026, acc=97.50% | ASR=11.80%
[backdoor | f=0.50 | round 09] clean test loss=0.1035, acc=97.40% | ASR=12.96%
[backdoor | f=0.50 | round 10] clean test loss=0.1048, acc=97.30% | ASR=13.87%
[backdoor | f=0.50 

Client 03 backdoor-poisoned train embeddings:   0%|          | 0/4 [00:00<?, ?it/s]

Client 06 backdoor-poisoned train embeddings:   0%|          | 0/5 [00:00<?, ?it/s]

Client 07 backdoor-poisoned train embeddings:   0%|          | 0/5 [00:00<?, ?it/s]

Malicious clients: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Total poisoned local train samples: 2462
Total malicious-client train samples: 12310
[backdoor | f=1.00 | round 00] clean test loss=0.0925, acc=97.66% | ASR=0.33%
[backdoor | f=1.00 | round 01] clean test loss=0.0978, acc=97.64% | ASR=0.87%
[backdoor | f=1.00 | round 02] clean test loss=0.1085, acc=97.58% | ASR=5.62%
[backdoor | f=1.00 | round 03] clean test loss=0.1211, acc=97.26% | ASR=16.27%
[backdoor | f=1.00 | round 04] clean test loss=0.1305, acc=96.88% | ASR=20.33%
[backdoor | f=1.00 | round 05] clean test loss=0.1404, acc=96.46% | ASR=22.91%
[backdoor | f=1.00 | round 06] clean test loss=0.1433, acc=96.24% | ASR=24.22%
[backdoor | f=1.00 | round 07] clean test loss=0.1432, acc=96.18% | ASR=25.33%
[backdoor | f=1.00 | round 08] clean test loss=0.1456, acc=96.02% | ASR=26.67%
[backdoor | f=1.00 | round 09] clean test loss=0.1423, acc=95.98% | ASR=27.40%
[backdoor | f=1.00 | round 10] clean test loss=0.1424, acc=95.88% | ASR=28.64%